## Note on data inputs

This notebook documents the **analysis method**. The raw licensed-facility inputs it reads (California Community Care Licensing / CDPH files containing facility administrator names, telephone numbers, and complaint histories) are **not distributed** with this repository, to protect vulnerable populations. De-identified and aggregated outputs are included; see `docs/DATA_AVAILABILITY.md`.

_As a result, this notebook is provided for transparency of method and will not run end-to-end against this public repository alone._

In [ ]:
06_adult_residential_facilities.ipynb

In [ ]:
  Cell In[1], line 1
    06_adult_residential_facilities.ipynb
      ^
SyntaxError: invalid decimal literal


In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

ROOT = Path("..")

DATA = ROOT / "data"

RAW = DATA / "raw"
PROCESSED = DATA / "processed"

In [ ]:
adult_res = pd.read_csv(
    RAW / "socialservices" / "AdultResidentialFacilities05172026.csv",
    usecols=range(38),
    names=main_cols,
    header=0
)

In [ ]:
main_cols = [
    "Facility Type",
    "Facility Number",
    "Facility Name",
    "Licensee",
    "Facility Administrator",
    "Facility Telephone Number",
    "Facility Address",
    "Facility City",
    "Facility State",
    "Facility Zip",
    "County Name",
    "Regional Office",
    "Facility Capacity",
    "Facility Status",
    "License First Date",
    "Closed Date",
    "Last Visit Date",
    "Inspection Visits",
    "Complaint Visits",
    "Other Visits",
    "Total Visits",
    "Citation Numbers",
    "POC Dates",
    "All Visit Dates",
    "Inspection Visit Dates",
    "Inspect TypeA",
    "Inspect TypeB",
    "Other Visit Dates",
    "Other TypeA",
    "Other TypeB",
    "Complaint Type A",
    "Complaint Type B",
    "Total Allegations",
    "Inconclusive Allegations",
    "Substantiated Allegations",
    "Unsubstantiated Allegations",
    "Unfounded Allegations",
    "Complaint Info"
]

In [ ]:
adult_res = pd.read_csv(
    RAW / "socialservices" / "AdultResidentialFacilities05172026.csv",
    usecols=range(38),
    names=main_cols,
    header=0
)

In [ ]:
adult_res.head()

In [ ]:
adult_res.shape

In [ ]:
adult_res["Facility Type"].value_counts()

In [ ]:
adult_res_eaton = adult_res[
    adult_res["Facility City"]
    .astype(str)
    .str.contains(
        "SIERRA|PASADENA|ALTADENA|CANADA",
        case=False,
        na=False
    )
].copy()

In [ ]:
len(adult_res_eaton)

In [ ]:
adult_res_eaton[
    [
        "Facility Name",
        "Facility City",
        "Facility Capacity",
        "Facility Status",
        "Closed Date",
        "Facility Address"
    ]
]

In [ ]:
adult_res_eaton["full_address"] = (
    adult_res_eaton["Facility Address"].astype(str)
    + ", "
    + adult_res_eaton["Facility City"].astype(str)
    + ", CA "
    + adult_res_eaton["Facility Zip"].astype(str)
)

In [ ]:
adult_res_eaton[
    [
        "Facility Name",
        "Facility Type",
        "full_address"
    ]
].head(20)

In [ ]:
adult_res_eaton[
    [
        "Facility Number",
        "Facility Address",
        "Facility City",
        "Facility State",
        "Facility Zip"
    ]
].to_csv(
    PROCESSED / "socialservices" / "adult_res_eaton_census_input.csv",
    index=False,
    header=False
)

In [ ]:
adult_res_eaton_census_input.csv

In [ ]:
data/processed/socialservices/adult_res_eaton_census_input.csv

In [ ]:
import os

os.listdir(PROCESSED / "socialservices")

In [ ]:
import os

os.listdir(PROCESSED / "socialservices")

In [ ]:
adult_res_geo = pd.read_csv(
    PROCESSED / "socialservices" / "adult_res_eaton_geocoded.csv",
    header=None
)

adult_res_geo.head()

In [ ]:
adult_res_geo.columns = [
    "Facility Number",
    "input_address",
    "match_status",
    "match_type",
    "matched_address",
    "coordinates",
    "tiger_line_id",
    "side",
    "state_fips",
    "county_fips",
    "tract",
    "block"
]

In [ ]:
adult_res_geo[["longitude", "latitude"]] = (
    adult_res_geo["coordinates"]
    .str.split(",", expand=True)
    .astype(float)
)

In [ ]:
adult_res_geocoded = adult_res_eaton.merge(
    adult_res_geo[
        [
            "Facility Number",
            "match_status",
            "match_type",
            "matched_address",
            "longitude",
            "latitude"
        ]
    ],
    on="Facility Number",
    how="left"
)

In [ ]:
adult_res_gdf = gpd.GeoDataFrame(
    adult_res_geocoded,
    geometry=gpd.points_from_xy(
        adult_res_geocoded["longitude"],
        adult_res_geocoded["latitude"]
    ),
    crs="EPSG:4326"
)

In [ ]:
perimeter = gpd.read_file(
    RAW / "perimeters" / "eaton_perimeter.geojson"
)

In [ ]:
adult_res_gdf = adult_res_gdf.to_crs(perimeter.crs)

adult_res_in_fire = gpd.sjoin(
    adult_res_gdf,
    perimeter,
    predicate="intersects"
)

len(adult_res_in_fire)

In [ ]:
impacted = pd.read_csv(
    PROCESSED / "calfire" / "eaton_impacted_structures.csv"
)

In [ ]:
impacted_gdf = gpd.GeoDataFrame(
    impacted,
    geometry=gpd.points_from_xy(
        impacted["Longitude"],
        impacted["Latitude"]
    ),
    crs="EPSG:4326"
)

In [ ]:
impacted_3857 = impacted_gdf.to_crs("EPSG:3857")

In [ ]:
adult_res_buffer = adult_res_gdf.to_crs("EPSG:3857").copy()

adult_res_buffer["geometry"] = (
    adult_res_buffer.geometry.buffer(50)
)

In [ ]:
adult_res_impacts = gpd.sjoin(
    impacted_3857,
    adult_res_buffer,
    predicate="intersects"
)

In [ ]:
len(adult_res_impacts)

In [ ]:
adult_res_impacts[
    [
        "Facility Name",
        "Facility Type",
        "Facility Capacity",
        "* Damage",
        "Site Address (parcel)"
    ]
]

In [ ]:
117 impacted structure matches

associated with:

adult residential/group-home facilities.

And the pattern is striking:

concentrated in Altadena
overwhelmingly destroyed
many small-capacity homes (4–8 residents)
repeated multi-structure campus impacts

In [ ]:
adult_res_impacts["Facility Name"].nunique()

In [ ]:
adult_res_major = adult_res_impacts[
    adult_res_impacts["* Damage"].isin([
        "Major (25-50%)",
        "Destroyed (>50%)"
    ])
]

In [ ]:
adult_res_major["Facility Name"].nunique()

In [ ]:
adult_res_major[
    [
        "Facility Name",
        "Facility Capacity"
    ]
].drop_duplicates()["Facility Capacity"].sum()

In [ ]:
adult_res_major[
    [
        "Facility Name",
        "Facility Type",
        "Facility Capacity",
        "* Damage"
    ]
].drop_duplicates()

In [ ]:
adult_res_export = (
    adult_res_major[
        [
            "Facility Name",
            "Facility Number",
            "Facility Type",
            "Facility Capacity",
            "Facility Address",
            "Facility City",
            "Facility Status",
            "Closed Date",
            "latitude",
            "longitude",
            "* Damage"
        ]
    ]
    .drop_duplicates(subset=["Facility Name"])
)

In [ ]:
adult_res_export.to_csv(
    PROCESSED / "socialservices" / "adult_residential_impacted_facilities.csv",
    index=False
)

In [ ]:
adult_res_export_gdf = gpd.GeoDataFrame(
    adult_res_export,
    geometry=gpd.points_from_xy(
        adult_res_export["longitude"],
        adult_res_export["latitude"]
    ),
    crs="EPSG:4326"
)

adult_res_export_gdf.to_file(
    PROCESSED / "socialservices" / "adult_residential_impacted_facilities.geojson",
    driver="GeoJSON"
)

In [ ]:
adult_res_major = adult_res_impacts[
    adult_res_impacts["* Damage"].isin([
        "Major (25-50%)",
        "Destroyed (>50%)"
    ])
]

adult_res_major["Facility Name"].nunique()

In [ ]:
adult_res_destroyed = adult_res_impacts[
    adult_res_impacts["* Damage"]
    == "Destroyed (>50%)"
]

adult_res_destroyed["Facility Name"].nunique()

In [ ]:
adult_res_major[
    [
        "Facility Name",
        "Facility Capacity"
    ]
].drop_duplicates()["Facility Capacity"].sum()

In [ ]:
adult_res_destroyed[
    [
        "Facility Name",
        "Facility Capacity"
    ]
].drop_duplicates()["Facility Capacity"].sum()

In [ ]:
adult_res_export = (
    adult_res_destroyed[
        [
            "Facility Name",
            "Facility Number",
            "Facility Type",
            "Facility Capacity",
            "Facility Address",
            "Facility City",
            "Facility Status",
            "Closed Date",
            "latitude",
            "longitude",
            "* Damage"
        ]
    ]
    .drop_duplicates(subset=["Facility Name"])
)

In [ ]:
adult_res_export.to_csv(
    PROCESSED / "socialservices" / "adult_residential_destroyed_facilities.csv",
    index=False
)

In [ ]:
adult_res_export_gdf = gpd.GeoDataFrame(
    adult_res_export,
    geometry=gpd.points_from_xy(
        adult_res_export["longitude"],
        adult_res_export["latitude"]
    ),
    crs="EPSG:4326"
)

adult_res_export_gdf.to_file(
    PROCESSED / "socialservices" / "adult_residential_destroyed_facilities.geojson",
    driver="GeoJSON"
)

In [ ]:
adult_res_impacts[
    [
        "Facility Name",
        "Facility Address",
        "Site Address (parcel)",
        "* Damage"
    ]
].drop_duplicates().head(50)

In [ ]:
adult_res_in_fire["address_clean"] = (
    adult_res_in_fire["Facility Address"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace(" STREET", " ST", regex=False)
    .str.replace(" AVENUE", " AVE", regex=False)
    .str.replace(" DRIVE", " DR", regex=False)
    .str.replace(" ROAD", " RD", regex=False)
    .str.strip()
)

In [ ]:
impacted["address_clean"] = (
    impacted["Site Address (parcel)"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace(" STREET", " ST", regex=False)
    .str.replace(" AVENUE", " AVE", regex=False)
    .str.replace(" DRIVE", " DR", regex=False)
    .str.replace(" ROAD", " RD", regex=False)
    .str.strip()
)

In [ ]:
impacted["street_only"] = (
    impacted["address_clean"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [ ]:
adult_res_exact = adult_res_in_fire.merge(
    impacted,
    left_on="address_clean",
    right_on="street_only",
    how="inner",
    suffixes=("_facility", "_calfire")
)

In [ ]:
adult_res_exact[
    [
        "Facility Name",
        "Facility Address",
        "Site Address (parcel)",
        "* Damage"
    ]
].drop_duplicates()

In [ ]:
adult_res_exact_major["Facility Name"].nunique()

In [ ]:
adult_res_exact_major = adult_res_exact[
    adult_res_exact["* Damage"].isin([
        "Major (25-50%)",
        "Destroyed (>50%)"
    ])
]

In [ ]:
adult_res_exact_major["Facility Name"].nunique()

In [ ]:
adult_res_exact_destroyed["Facility Name"].nunique()

In [ ]:
adult_res_exact_destroyed = adult_res_exact[
    adult_res_exact["* Damage"]
    == "Destroyed (>50%)"
]

In [ ]:
adult_res_exact_destroyed["Facility Name"].nunique()

In [ ]:
adult_res_impacted["Facility Name"].nunique()

In [ ]:
adult_res_major["Facility Name"].nunique()

In [ ]:
adult_res_impacted = adult_res_exact[
    adult_res_exact["* Damage"] != "No Damage"
]

In [ ]:
adult_res_impacted["* Damage"].value_counts()

In [ ]:
adult_res_impacted["Facility Name"].nunique()

In [ ]:
adult_res_major = adult_res_exact[
    adult_res_exact["* Damage"].isin([
        "Major (25-50%)",
        "Destroyed (>50%)"
    ])
]

adult_res_destroyed = adult_res_exact[
    adult_res_exact["* Damage"] == "Destroyed (>50%)"
]

print("Impacted:", adult_res_impacted["Facility Name"].nunique())
print("Major/destroyed:", adult_res_major["Facility Name"].nunique())
print("Destroyed:", adult_res_destroyed["Facility Name"].nunique())